# Colab runner for the sampler chains

Runs the expensive chains on a rented GPU and hands back the `.npz` files the
local notebook loads. Use it for anything that hurts on the laptop: the CNN
full-batch chains, longer MLP chains, bigger minibatch sweeps.

Before running: **Runtime -> Change runtime type -> GPU** (T4 is fine, A100 is
much faster for the CNN).

In [ ]:
!nvidia-smi

## Get the code

The repo is private, so cloning needs a GitHub token (create a fine-grained
token with read access to the repo, paste it below). Alternative: zip the
repo locally and upload it via the Files panel on the left.

In [ ]:
GITHUB_TOKEN = ""   # paste a fine-grained read token here
REPO = "YasirAlsugair/ray-tracing-sampler-experiments"

if GITHUB_TOKEN:
    !git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git repo
else:
    print("no token set: upload a zip of the repo and unzip it to ./repo instead")
%cd repo

## Run a chain

Same commands as on the laptop; the scripts pick CUDA automatically. Colab
already ships torch and torchvision, and MNIST downloads itself. Edit the
command for what you need:

- exact CNN chain, longer than the laptop could afford:
  `run cnn 1.5e-4 30 5000`
- flat-prior variant: append `flat`
- the minibatch sweep: `experiments/exp6_minibatch.py`

One caution: free Colab disconnects idle sessions and caps runtime around 12
hours, so keep single runs under a few hours (the printout at the end is the
sign the npz was written).

In [ ]:
# For strict comparability with the laptop runs: upload the local
# checkpoints (results/checkpoints/exp6_mlp.pt and exp6_cnn.pt, a few
# hundred KB) via the Files panel INSTEAD of retraining here. Retraining
# on this machine gives a different starting point (different hardware
# and random streams), which is fine for new experiments but not for
# extending existing chains.
# !python experiments/exp6_simple_mnist_train.py
!python experiments/exp6_sample_metropolis.py run cnn 1.5e-4 30 5000

## Bring the results home

Download the chain files, then drop them into `empirical/results/tables/` on
the laptop and re-run the analysis notebook; it picks them up by filename.

In [ ]:
from google.colab import files
import glob
for path in glob.glob("results/tables/exp6_rt_chain_*.npz") + glob.glob("results/tables/exp6_mb_*.npz"):
    print(path)
    files.download(path)

One honesty note for writeups: chains produced on CUDA are statistically
equivalent but not bit-identical to MPS runs (different random streams and
kernel arithmetic), so keep a note of which device produced which artifact.
The npz files already record their settings.